# Afterstates, shared patterns, and planning

The original Q-table memorized entire boards. An n-tuple model instead learns values of local tile patterns and sums them. Eight rotations/reflections share weights. This notebook does small calculations; saved long experiments are read from disk.

Execution path: `game.py` defines the rules → `fast2048.py` compiles the same rules and updates → `agents/ntuple.py` supplies patterns and weights → `ntuple_train.py` collects games → `evaluate.py` / `view.py` evaluate and replay.

```mermaid
flowchart LR
 S[Current board] --> M[Choose legal move]
 M --> R[Merge reward]
 M --> X[Afterstate before spawn]
 X --> N[Random tile]
 N --> S2[Next board]
 X --> V[N-tuple value]
```

Sources: [Guei dissertation](https://arxiv.org/abs/2212.11087), [optimistic TD](https://arxiv.org/abs/2111.11090), [Jaskowski](https://arxiv.org/abs/1604.05085).

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'rl2048').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
from rl2048.agents.ntuple import NTupleAgent, encode, row_tables
from rl2048.fast2048 import value, update_value
from rl2048.game import move, legal_actions
board = np.array([[2,2,4,0],[0,4,0,0],[0,0,0,0],[0,0,0,0]])
after, reward, _ = move(board, 3)
print('Before:\n',board,'\nAfter left, before spawn:\n',after,'\nCurrent reward:',reward)


## Follow one update

Action selection uses $r_t+V(x_t)$. But $V(x_t)$ begins **after** the current reward, so its TD error is

$$\delta_t=r_{t+1}+\gamma V(x_{t+1})-V(x_t).$$

Suppose $r_{t+1}=8$, the next afterstate has value 20, the current value is 0, and $\gamma=1$. The target is 28. With learning rate 0.1 and 32 active features, each feature occurrence receives $0.1\times28/32=0.0875$.

A feature can occur more than once because a board has repeated patterns. The implementation accumulates each occurrence; therefore total predicted-value change can exceed $0.1\delta$ when there are duplicates.

In [ ]:
agent = NTupleAgent('4x4')  # tiny demonstration tables, same update kernel
state = encode(after)
before = value(state, agent.weights, agent.patterns)
delta = 8 + 20 - before
active_features = agent.patterns.shape[0] * agent.patterns.shape[1]
dummy = np.zeros((1,1),np.float32)
update_value(state, 28., agent.weights, agent.patterns, .1, dummy, dummy, False)
print({'old_value':before,'target':28,'TD_error':delta,'update_per_occurrence':.1*delta/active_features,
       'new_value':value(state,agent.weights,agent.patterns)})

## Why planning helps

At a chance node, enumerate every empty square and both possible tiles. Each square gets $0.9/n$ probability for a 2 and $0.1/n$ for a 4. At a decision node, choose the legal action with the largest immediate reward plus continuation value. Search depth 1 uses the learned value immediately; depth 2 simulates one more random spawn and decision.

Discount 1 targets total game score. Randomness does not require a smaller or larger discount: it requires an expectation. Deeper search costs more computation and still depends on the leaf value estimates.

In [ ]:
import json
for label, path in [('local model',ROOT/'runs/research/native_otd/validation.json'),
                    ('published reference',ROOT/'runs/research/published/checkpoint/validation.json')]:
    if path.exists():
        print(label)
        for depth, result in json.loads(path.read_text()).items():
            s=result['summary']
            print('depth',depth,'mean score',round(s['mean_score']), 'games',s['games'],'seconds',round(s['seconds'],2))

## What the short experiment taught us

The zero-initialized 4×6 model beat the larger optimistic candidates in our first short-budget screen. That is a budget-specific finding, not a contradiction of optimistic TD's long-run results. Temporal coherence improved greedy play but did not automatically improve shallow search.

Published pretrained results use weights learned elsewhere. Local checkpoints and download provenance remain separate. The graph below may change as experiments finish.

In [ ]:
from IPython.display import Image, display, HTML
plot = ROOT/'runs/research/training_overview.png'
if plot.exists(): display(Image(filename=str(plot)))
display(HTML('<a href="http://127.0.0.1:8848/research/local_best_replay.html" target="_blank">Replay our locally trained agent</a>'))